# Lab 21: The Attention Spotlight

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 3, *The Attention Filter*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-21.ipynb)

**What you will do:** measure how much faster you spot a letter when you were correctly told where to attend than when you were told the wrong corner, and see the same content-dependent reweighting in a transformer's self-attention.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 21 you fixed your gaze on a central cross while a friend told you which corner to
attend to. You identified letters in that corner faster and more accurately than letters in
the other corners -- even though your eyes never moved. Attention is not the same as gaze:
you can spotlight a location in your mind while looking somewhere else, and spotlighting one
location makes the others harder to see.

## Record your results

A stopwatch is too coarse to time a single flashed letter, so use the smartphone extension's
attention-test app. Run at least 10 trials in which the cue correctly names the corner the
target then appears in ("valid"), and at least 10 in which the cue names a different corner
("invalid"). Record your mean reaction time and accuracy for each condition.

In [ ]:
n_trials = 10
my_rt_ms = {"valid cue": 410, "invalid cue": 460}          # example data: replace with your app's averages
my_accuracy = {"valid cue": 0.95, "invalid cue": 0.80}     # example data: proportion correct

results = pd.DataFrame({"mean RT (ms)": my_rt_ms, "accuracy": my_accuracy})
display(results)
results["mean RT (ms)"].plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3.5))
plt.ylabel("mean reaction time (ms)"); plt.xticks(rotation=0); plt.show()

## Compare

The book does not give a specific millisecond figure for the validity effect, only its
direction: a valid cue should speed you up, and an invalid cue should slow you down, even
though your gaze never left the cross (Posner, 1980).

In [ ]:
validity_effect = results.loc["invalid cue", "mean RT (ms)"] - results.loc["valid cue", "mean RT (ms)"]
print(f"Your validity effect: {validity_effect:.0f} ms slower for invalid than for valid cues")
if validity_effect > 0:
    print("Positive, as predicted: a valid cue speeds responses and an invalid cue slows them, even"
          " though your eyes stayed on the central cross the whole time.")
else:
    print("Your invalid-cue trials were not slower on average. With only a handful of trials, chance"
          " variation can hide a real effect -- try more trials, or double-check that your gaze really"
          " stayed fixed on the cross.")

## The AI side

The book compares the brain's attentional spotlight to the self-attention mechanism inside a
transformer (Vaswani et al., 2017): a word's meaning is computed by weighting the other words
it "attends to," and those weights change with context, just as the parietal cortex can
redirect the sensory gain to a different corner on your next trial. Below, a small pretrained
language model reads the word "bank" in two sentences and shows which other words it leans on
to interpret it.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, attn_implementation="eager", output_attentions=True)
model.eval()

sentences = {
    "river context": "I sat on the bank and watched the fish swim past.",
    "money context": "I walked into the bank to withdraw some cash.",
}

for label, sentence in sentences.items():
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    bank_idx = tokens.index("bank")
    with torch.no_grad():
        attentions = model(**inputs).attentions   # one tensor per layer: (batch, heads, query, key)
    last_layer = attentions[-1][0]                 # (heads, query, key)
    attn_from_bank = last_layer[:, bank_idx, :].mean(dim=0)  # averaged over heads
    top = attn_from_bank.argsort(descending=True)[:5]
    print(f"\n[{label}] \"{sentence}\"")
    for i in top:
        print(f"  'bank' attends to {tokens[i]!r:>10s}  (weight {attn_from_bank[i]:.2f})")

In the river sentence, "bank" should lean on words like "sat" or "fish"; in the money
sentence, on "withdraw" or "cash." That context-dependent reweighting is the same kind of
move the parietal cortex makes when it changes which corner's sensory gain is turned up. The
analogy has real limits: the transformer recomputes its attention weights over the whole
sentence in a single forward pass, with no gaze, no time course, and no zero-sum trade-off
between locations -- it can attend strongly to several words at once without "spending"
attention it takes from elsewhere.

## Pool the class data

A single person's validity effect is noisy over 10-20 trials. Pool the class: each person
adds one row (an anonymous ID and their valid/invalid mean RTs in milliseconds), downloads
it as CSV, and loads it here. The rows below are placeholders so the cell runs.

In [ ]:
import io
csv_text = """id,valid_ms,invalid_ms
P01,395,430
P02,420,470
P03,380,410
P04,440,480
P05,405,455
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
df["validity_effect_ms"] = df.invalid_ms - df.valid_ms
boot = [df.validity_effect_ms.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean validity effect: {df.validity_effect_ms.mean():.0f} ms "
      f"(95% bootstrap interval {lo:.0f}-{hi:.0f} ms, n = {len(df)})")
df.validity_effect_ms.plot.hist(bins=8, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvline(0, color="grey", linestyle="--")
plt.xlabel("validity effect (ms, invalid - valid)"); plt.show()

## Questions to think about

1. Your validity effect (if positive) shows you can shift attention without moving your eyes. Which two brain regions does the book credit with directing this covert shift?
2. Step 4 calls attention "zero-sum" -- enhancing one corner suppresses the others. Did your invalid-cue accuracy drop as well as your RT increasing? What would a strictly zero-sum system predict about accuracy at the uncued corners?
3. In the AI Lab, "bank" attended to different words depending on context. What is the closest human equivalent, in the book's account of parietal-cortex control over V1 gain?
4. The zoom-lens model (Eriksen & St James, 1986) says the spotlight can widen or narrow. If a cue pointed to two corners at once instead of one, would you predict a faster or slower validly cued RT than cueing a single corner? Why?

## Challenge

Test the "zero-sum" claim directly. Cue one corner, but present the target at one of the
*other three* corners instead of the cued one. Run enough trials to get an accuracy for each
of the three uncued corners. Is accuracy roughly equal across all three, or lower specifically
at the corner diagonally opposite the cue?

In [ ]:
# Example data: replace with your own accuracy (proportion correct) at each uncued corner,
# when the cue pointed to a different, fixed corner each time.
my_uncued_accuracy = {"adjacent corner 1": 0.70, "adjacent corner 2": 0.72,
                       "diagonally opposite": 0.55}  # example data

results = pd.Series(my_uncued_accuracy, name="accuracy")
display(results.to_frame())
results.plot.bar(color="firebrick", figsize=(5, 3)); plt.ylim(0, 1); plt.show()
print("If the diagonally opposite corner is noticeably worse than the two adjacent corners,")
print("suppression is graded by distance from the cue, not a simple on/off switch.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-21.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")